# Feature Engineering

## Objetivo
Construir el conjunto de features (X) y unirlas con la variable objetivo (y)
para generar el dataset maestro listo para el modelado.

## Estructura

### Bloque A — Features de metadatos (ya limpias del Notebook 02)
- Cargar `games_clean.parquet` directamente (tags, géneros, precio, etc.)

### Bloque B — Features NLP desde reseñas tempranas (T+30 días)
- B1. Filtrar reseñas de los primeros 30 días desde el lanzamiento
- B2. Feature: ratio de positivos tempranos (`early_positive_ratio`)
- B3. Feature: longitud media de reseñas (`avg_review_len_chars`)
- B4. Feature: ratio de reseñas con menciones a bugs (`bug_mention_ratio`)
- B5. Feature: análisis de sentimiento con VADER (`avg_sentiment_score`)
- B6. Agregación por juego

### Bloque C — Features de Twitch
- Cargar `twitch_features.parquet` directamente

### Bloque D — Construcción del dataset maestro
- Unir todas las fuentes por `appid`
- Unir con `target.parquet`
- Verificar nulos y shape final
- Guardar `dataset_maestro.parquet`


In [1]:
import pandas as pd
from pathlib import Path

PROCESSED = Path('../data/processed/')

df_games   = pd.read_parquet(PROCESSED / 'games_clean.parquet')
df_reviews = pd.read_parquet(PROCESSED / 'reviews_clean.parquet')
df_target  = pd.read_parquet(PROCESSED / 'target.parquet')
df_twitch  = pd.read_parquet(PROCESSED / 'twitch_features.parquet')

print("=== games_clean ===")
print(f"Shape: {df_games.shape}")
print(f"Columnas clave: {['appid','name','release_date','price_clean','peak_ccu','median_playtime_forever']}")

print("\n=== reviews_clean ===")
print(f"Shape: {df_reviews.shape}")
print(f"Columnas: {df_reviews.columns.tolist()}")
print(f"Rango post_date: {df_reviews['post_date'].min()} → {df_reviews['post_date'].max()}")

print("\n=== target ===")
print(f"Shape: {df_target.shape}")
print(df_target['is_successful'].value_counts())

print("\n=== twitch_features ===")
print(f"Shape: {df_twitch.shape}")
print(f"Juegos con Twitch=1: {df_twitch['appeared_on_twitch_top200'].sum()}")

# Verificar que release_date está disponible para el filtro temporal
print("\n=== Verificación release_date ===")
print(f"Nulos en release_date: {df_games['release_date'].isna().sum()}")
print(f"Tipo: {df_games['release_date'].dtype}")
print(df_games[['appid','name','release_date']].head(3))


=== games_clean ===
Shape: (11507, 57)
Columnas clave: ['appid', 'name', 'release_date', 'price_clean', 'peak_ccu', 'median_playtime_forever']

=== reviews_clean ===
Shape: (3745380, 8)
Columnas: ['user', 'playtime', 'post_date', 'helpfulness', 'review', 'recommend', 'appid', 'review_len_chars']
Rango post_date: 2011-04-17 00:00:00 → 2024-12-06 00:00:00

=== target ===
Shape: (11371, 5)
is_successful
0    8072
1    3299
Name: count, dtype: int64

=== twitch_features ===
Shape: (11507, 6)
Juegos con Twitch=1: 356

=== Verificación release_date ===
Nulos en release_date: 0
Tipo: datetime64[us]
     appid              name release_date
0  2218750  Halls of Torment   2024-09-24
1  1089630     奇迹一刻 Surmount   2022-02-13
2   999220  Amnesia: Rebirth   2020-10-20


# Bloque B - Features NLP desde reseñas tempranas (T+30 días)

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path('../data/processed/')

df_games   = pd.read_parquet(PROCESSED / 'games_clean.parquet')
df_reviews = pd.read_parquet(PROCESSED / 'reviews_clean.parquet')

# Cruzar reseñas con release_date del juego
df_reviews = df_reviews.merge(
    df_games[['appid', 'release_date']], 
    on='appid', 
    how='left'
)

# Calcular días desde el lanzamiento
df_reviews['days_since_release'] = (
    df_reviews['post_date'] - df_reviews['release_date']
).dt.days

# Filtrar solo reseñas de los primeros 30 días
df_early = df_reviews[
    (df_reviews['days_since_release'] >= 0) & 
    (df_reviews['days_since_release'] <= 30)
].copy()

print(f"Total reseñas: {len(df_reviews):,}")
print(f"Reseñas en primeros 30 días (T+30): {len(df_early):,}")
print(f"Porcentaje sobre el total: {len(df_early)/len(df_reviews)*100:.1f}%")
print(f"Juegos con al menos 1 reseña temprana: {df_early['appid'].nunique():,}")
print(f"Juegos sin ninguna reseña en T+30: {11507 - df_early['appid'].nunique():,}")

print(f"\nDistribución de días desde lanzamiento (T+30):")
print(df_early['days_since_release'].describe())


Total reseñas: 3,745,380
Reseñas en primeros 30 días (T+30): 1,317,024
Porcentaje sobre el total: 35.2%
Juegos con al menos 1 reseña temprana: 11,467
Juegos sin ninguna reseña en T+30: 40

Distribución de días desde lanzamiento (T+30):
count    1.317024e+06
mean     6.912033e+00
std      7.771709e+00
min      0.000000e+00
25%      1.000000e+00
50%      4.000000e+00
75%      1.000000e+01
max      3.000000e+01
Name: days_since_release, dtype: float64


In [3]:
# ── Diccionario de términos técnicos negativos (bugs, crashes) ───
BUG_KEYWORDS = [
    'bug', 'crash', 'broken', 'fix', 'glitch', 'error', 'freeze',
    'lag', 'stutter', 'fps', 'unplayable', 'refund', 'broken',
    'patch', 'issue', 'problem', 'not working', 'doesnt work'
]

def contiene_bug_keywords(texto):
    texto_lower = str(texto).lower()
    return int(any(kw in texto_lower for kw in BUG_KEYWORDS))

# ── VADER para análisis de sentimiento ───────────────────────────
# VADER funciona bien para texto en inglés y es muy rápido
try:
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    import nltk
    nltk.download('vader_lexicon', quiet=True)
    sia = SentimentIntensityAnalyzer()
    def get_sentiment(texto):
        try:
            return sia.polarity_scores(str(texto))['compound']
        except:
            return 0.0
    usar_vader = True
    print("✅ VADER disponible")
except ImportError:
    usar_vader = False
    print("⚠️  VADER no disponible, instala nltk: pip install nltk")

# ── Aplicar features a nivel de reseña individual ────────────────
print("Calculando features por reseña...")

df_early['has_bug_mention'] = df_early['review'].apply(contiene_bug_keywords)

if usar_vader:
    # Aplicar VADER solo a reseñas en inglés estimado (< 500 chars para velocidad)
    # Para reseñas largas truncamos a 500 chars (VADER no mejora con más texto)
    df_early['sentiment_score'] = df_early['review'].apply(
        lambda x: get_sentiment(str(x)[:500])
    )
else:
    df_early['sentiment_score'] = 0.0

print("✅ Features por reseña calculadas")
print(df_early[['has_bug_mention', 'sentiment_score', 'review_len_chars']].describe())


✅ VADER disponible
Calculando features por reseña...
✅ Features por reseña calculadas
       has_bug_mention  sentiment_score  review_len_chars
count     1.317024e+06     1.317024e+06      1.317024e+06
mean      1.525849e-01     1.935584e-01      4.568999e+02
std       3.595871e-01     4.698936e-01      8.547675e+02
min       0.000000e+00    -9.999000e-01      1.000000e+01
25%       0.000000e+00     0.000000e+00      5.100000e+01
50%       0.000000e+00     0.000000e+00      1.570000e+02
75%       0.000000e+00     6.310000e-01      4.650000e+02
max       1.000000e+00     9.999000e-01      8.000000e+03


In [4]:
# Agregar todas las features NLP por juego
df_nlp_features = df_early.groupby('appid').agg(
    # Volumen de reseñas tempranas
    early_review_count      = ('recommend', 'count'),
    
    # Recepción temprana
    early_positive_ratio    = ('recommend', 'mean'),
    
    # Longitud de reseñas (proxy de engagement)
    avg_review_len_chars    = ('review_len_chars', 'mean'),
    median_review_len_chars = ('review_len_chars', 'median'),
    
    # Bugs y problemas técnicos
    bug_mention_ratio       = ('has_bug_mention', 'mean'),
    
    # Sentimiento VADER
    avg_sentiment_score     = ('sentiment_score', 'mean'),
    std_sentiment_score     = ('sentiment_score', 'std'),
    
    # Helpfulness temprana
    avg_early_helpfulness   = ('helpfulness', 'mean'),
    
    # Playtime de los primeros reviewers
    avg_early_playtime      = ('playtime', 'mean'),
    
).reset_index()

# Rellenar std_sentiment NaN (juegos con solo 1 reseña)
df_nlp_features['std_sentiment_score'] = df_nlp_features['std_sentiment_score'].fillna(0)

print(f" Features NLP agregadas por juego")
print(f"   Juegos con features NLP: {len(df_nlp_features):,}")
print(f"\nEstadísticas:")
print(df_nlp_features.drop('appid', axis=1).describe())


 Features NLP agregadas por juego
   Juegos con features NLP: 11,467

Estadísticas:
       early_review_count  early_positive_ratio  avg_review_len_chars  \
count        11467.000000          11467.000000          11467.000000   
mean           114.853405              0.795229            440.494836   
std             97.719654              0.178257            278.514762   
min              1.000000              0.000000             13.000000   
25%             42.000000              0.706897            240.250301   
50%             85.000000              0.839506            375.428571   
75%            162.000000              0.931929            572.682857   
max            500.000000              1.000000           4426.000000   

       median_review_len_chars  bug_mention_ratio  avg_sentiment_score  \
count             11467.000000       11467.000000         11467.000000   
mean                202.189544           0.143512             0.224494   
std                 160.455650      

In [7]:
df_target  = pd.read_parquet(PROCESSED / 'target.parquet')
df_twitch  = pd.read_parquet(PROCESSED / 'twitch_features.parquet')

# Columnas de metadatos a conservar del games_clean
meta_cols = [
    'appid', 'release_date', 'required_age', 'price_clean', 'is_free',
    'is_software_tool', 'peak_ccu', 'median_playtime_forever',
    'owners_midpoint', 'num_languages', 'num_categories', 'has_full_audio',
    'csv_review_count'
] + [c for c in df_games.columns if c.startswith('genre_') or c.startswith('tag_')]

df_meta = df_games[meta_cols].copy()

# ── Unir todas las fuentes ────────────────────────────
df_master = df_meta.merge(df_nlp_features, on='appid', how='left')
df_master = df_master.merge(df_twitch, on='appid', how='left')
df_master = df_master.merge(
    df_target[['appid', 'is_successful']], on='appid', how='inner'
)

print(f"=== DATASET MAESTRO ===")
print(f"Shape: {df_master.shape}")
print(f"\nJuegos sin features NLP (sin reseñas en T+30): "
      f"{df_master['early_review_count'].isna().sum():,}")
print(f"\nNulos por columna (solo las que tienen nulos):")
nulos = df_master.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))
print(f"\nDistribución del target:")
print(df_master['is_successful'].value_counts())


=== DATASET MAESTRO ===
Shape: (11371, 71)

Juegos sin features NLP (sin reseñas en T+30): 25

Nulos por columna (solo las que tienen nulos):
early_review_count         25
early_positive_ratio       25
avg_review_len_chars       25
median_review_len_chars    25
bug_mention_ratio          25
avg_sentiment_score        25
std_sentiment_score        25
avg_early_helpfulness      25
avg_early_playtime         25
dtype: int64

Distribución del target:
is_successful
0    8072
1    3299
Name: count, dtype: int64


In [8]:
# Juegos sin reseñas en T+30 → imputar con 0 o mediana según el tipo
nlp_cols = [
    'early_review_count', 'early_positive_ratio', 'avg_review_len_chars',
    'median_review_len_chars', 'bug_mention_ratio', 'avg_sentiment_score',
    'std_sentiment_score', 'avg_early_helpfulness', 'avg_early_playtime'
]

# Para juegos sin reseñas tempranas: imputar con 0
# (sin señal temprana = peor escenario, interpretable)
for col in nlp_cols:
    if col in df_master.columns:
        df_master[col] = df_master[col].fillna(0)

# Twitch: ya imputado con 0 en el Notebook 02
# owners_midpoint: imputar con mediana
df_master['owners_midpoint'] = df_master['owners_midpoint'].fillna(
    df_master['owners_midpoint'].median()
)

# Verificación final
print(f"Nulos tras imputación:")
nulos_final = df_master.isnull().sum()
print(nulos_final[nulos_final > 0])

# Guardar
df_master.to_parquet(PROCESSED / 'dataset_maestro.parquet', index=False)
size_mb = (PROCESSED / 'dataset_maestro.parquet').stat().st_size / 1024 / 1024

print(f"\n✅ dataset_maestro.parquet guardado")
print(f"   Shape: {df_master.shape}")
print(f"   Tamaño: {size_mb:.2f} MB")
print(f"\nPrimeras filas:")
print(df_master[['appid', 'early_review_count', 'early_positive_ratio', 
                  'bug_mention_ratio', 'avg_sentiment_score', 
                  'appeared_on_twitch_top200', 'is_successful']].head(5))


Nulos tras imputación:
Series([], dtype: int64)

✅ dataset_maestro.parquet guardado
   Shape: (11371, 71)
   Tamaño: 0.88 MB

Primeras filas:
     appid  early_review_count  early_positive_ratio  bug_mention_ratio  \
0  2218750                48.0              0.791667           0.125000   
1  1089630                14.0              0.500000           0.071429   
2   999220                50.0              0.780000           0.360000   
3  1656780               210.0              0.657143           0.228571   
4  1477170                60.0              0.950000           0.083333   

   avg_sentiment_score  appeared_on_twitch_top200  is_successful  
0             0.158871                          0              0  
1             0.000000                          0              0  
2            -0.063474                          1              0  
3             0.147166                          0              0  
4             0.092008                          0              0  
